📓 Notebook Complet : Pinecone Serverless Reranking in Action


## Notebook: RAG System with Langchain, Hugging Face and FAISS

### Part 1: Setup and Installations

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Aggressively uninstall conflicting langchain and pydantic packages to ensure a clean slate
!pip uninstall -y langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface langchain-classic pydantic pydantic-settings > /dev/null 2>&1

# Install pydantic V2 first to ensure consistency with newer langchain versions
!pip install -qU pydantic>=2.0.0

# Install necessary libraries, allowing pip to resolve compatible versions, after pydantic is set
!pip install -qU langchain datasets faiss-cpu sentence-transformers transformers langchain-community langchain-text-splitters langchain-huggingface

print("✅ Libraries installed successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 14.6 MB/s eta 0:00:00
✅ Libraries installed successfully.


### Part 2: Data Loading and Preparation

In [ ]:
# Load the 'databricks/databricks-dolly-15k' dataset
from langchain_community.document_loaders import HuggingFaceDatasetLoader

dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "response"  # Column containing the text content

loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
documents = loader.load()

print(f"✅ Loaded {len(documents)} documents from {dataset_name}.")
print("First document example:")
print(documents[0].page_content[:500]) # Display first 500 characters of the first document

✅ Loaded 15011 documents from databricks/databricks-dolly-15k.
First document example:
"Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route."


In [ ]:
# Chunk documents using RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True,
)

chunks = text_splitter.split_documents(documents)

print(f"✅ Split {len(documents)} documents into {len(chunks)} chunks.")
print("First chunk example:")
print(chunks[0].page_content[:500])

NameError: name 'documents' is not defined

### Part 3: Embedding and Vector Store Creation

In [ ]:
# Load a Sentence Transformer model for embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Using 'all-MiniLM-L6-v2' as a common and efficient embedding model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

print(f"✅ Embedding model '{model_name}' loaded.")

RuntimeError: no validator found for <class 'langchain_core.outputs.chat_generation.ChatGeneration'>, see `arbitrary_types_allowed` in Config

In [ ]:
# Create FAISS vector store
from langchain_community.vectorstores import FAISS

# This step might take a few minutes depending on the number of chunks
vector_store = FAISS.from_documents(chunks, embeddings)

print("✅ FAISS vector store created and populated.")

NameError: name 'chunks' is not defined

### Part 4: Language Model (LLM) Setup

In [ ]:
# Load the 'Intel/dynamic_tinybert' LLM
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

# The LLM specified is Intel/dynamic_tinybert
# We will use it with a text-generation pipeline
model_id = "Intel/dynamic_tinybert"

# Initialize the text generation pipeline
pipeline = pipeline(
    "text-generation",
    model=model_id,
    tokenizer=model_id, # Use the same tokenizer as the model
    max_new_tokens=100, # Limit response length
    temperature=0.7,
    top_p=0.95,
    repetition_penalty=1.15,
)

llm = HuggingFacePipeline(pipeline=pipeline)

print(f"✅ LLM '{model_id}' loaded and pipeline initialized.")

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

Invalid model-index. Not loading eval results into CardData.
[transformers] If you want to use `BertLMHeadModel` as a standalone, add `is_decoder=True.`


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'repetition_penalty', 'temperature', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ LLM 'Intel/dynamic_tinybert' loaded and pipeline initialized.


/tmp/ipykernel_2054/1459436412.py:20: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipeline)


### Part 5: Building and Testing the RetrievalQA Chain

In [ ]:
# Build the RetrievalQA chain
from langchain.chains import RetrievalQA # Langchain core components like RetrievalQA are still often directly in langchain.chains

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", # 'stuff' combines all documents into a single prompt
    retriever=vector_store.as_retriever(),
    return_source_documents=True
)

print("✅ RetrievalQA chain built.")

TypeError: Cannot create a consistent method resolution
order (MRO) for bases Generic, ABC

In [ ]:
# Test the RAG system with a query
query = "What is the meaning of life?"

print(f"\n❓ Query: {query}")
response = qa_chain({"query": query})

print(f"\n💡 Answer: {response['result']}")
print("\n--- Source Documents ---")
for i, doc in enumerate(response['source_documents']):
    print(f"Document {i+1} (Source: {doc.metadata.get('source', 'N/A')}):")
    print(doc.page_content[:300], "...")
    print("\n")


❓ Query: What is the meaning of life?


NameError: name 'qa_chain' is not defined

In [ ]:
# Another query example
query_2 = "Explain machine learning in simple terms."

print(f"\n❓ Query: {query_2}")
response_2 = qa_chain({"query": query_2})

print(f"\n💡 Answer: {response_2['result']}")
print("\n--- Source Documents ---")
for i, doc in enumerate(response_2['source_documents']):
    print(f"Document {i+1} (Source: {doc.metadata.get('source', 'N/A')}):")
    print(doc.page_content[:300], "...")
    print("\n")


❓ Query: Explain machine learning in simple terms.


NameError: name 'qa_chain' is not defined

Partie 1 : Chargement des documents et exécution du reranking

Installation de Pinecone

In [ ]:
# PARTIE 1 - ÉTAPE 1 : Installation des bibliothèques Pinecone
# On installe la version 6.0.1 de Pinecone et les helpers pour notebooks.
# Le '!' permet d'exécuter la commande directement dans Colab.

!pip install -U pinecone==6.0.1 pinecone-notebooks


2 : Authentification avec Pinecone

In [ ]:
# PARTIE 1 - ÉTAPE 2 : Authentification
# Cette cellule va te demander ta clé API Pinecone si elle n'est pas déjà
# définie dans les variables d'environnement.
# Récupère ta clé depuis ton tableau de bord Pinecone (onglet "API Keys").

import os
if not os.environ.get("pcsk_6yag1G_QKyXJn3g5EgWaev33fd9kUqp4c1Q76ZJB7nXHnSk6Z38HPg3gYgvM8iMSEwDBeS"): #je dois metre ma clé
    from pinecone_notebooks.colab import Authenticate
    Authenticate()  # Une fenêtre te demandera de coller ta clé



3 : Instantiation du client Pinecone

In [ ]:
# PARTIE 1 - ÉTAPE 3 : Création du client
# Le client 'pc' est notre point d'entrée pour toutes les opérations
# (création d'index, requêtes, reranking).

from pinecone import Pinecone

api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)


4 : Définition de la requête et des documents

In [ ]:
# PARTIE 1 - ÉTAPE 4 : Création de la requête et des documents
# On prépare 5 documents : 2 sur la pomme (fruit) et 3 sur Apple (entreprise).
# Cela permet de tester la capacité du reranker à comprendre le contexte.

query = "Tell me about Apple's latest products"

documents = [
    "Apple is a sweet and crisp fruit, often red or green, grown on trees.",
    "Apple Inc. is a leading technology company known for the iPhone and MacBook.",
    "Granny Smith apples are particularly tart and great for baking pies.",
    "Apple recently released the new MacBook Pro with the M3 chip and improved battery life.",
    "Compared to oranges, apples have a higher fiber content and are eaten worldwide."
]

print("📝 Documents définis.")
print(f"Requête : {query}")

📝 Documents définis.
Requête : Tell me about Apple's latest products


 5 : Appel du reranker

In [ ]:
# PARTIE 1 - ÉTAPE 5 : Exécution du reranking
# On utilise le modèle 'bge-reranker-v2-m3' de Pinecone.
# On transforme les documents en une liste de dictionnaires avec 'id' et 'text'.
# top_n = 3 : on veut les 3 meilleurs résultats après reranking.

from pinecone import RerankModel

reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3  # On récupère seulement les 3 plus pertinents
)

print("✅ Reranking effectué avec succès.")

✅ Reranking effectué avec succès.


6 : Inspection des résultats rerankés

In [ ]:
# PARTIE 1 - ÉTAPE 6 : Affichage des résultats
# La réponse du reranker se trouve dans 'reranked.data'.
# Chaque élément a un attribut .score (similarité) et .document.text.

def show_reranked_results(query, matches):
    print(f"Query: {query}")
    print("-" * 60)
    for i, m in enumerate(matches):
        # On affiche le rang (i+1), le score et le texte du document
        print(f"{i+1}. Score: {m.score:.4f}")
        print(f"   Texte: {m.document.text}\n")

# 'reranked.data' contient la liste des résultats triés par pertinence
show_reranked_results(query, reranked.data)

print("💡 Analyse : Les documents sur Apple (entreprise) doivent avoir les scores les plus élevés.")

Query: Tell me about Apple's latest products
------------------------------------------------------------
1. Score: 0.3900
   Texte: Apple recently released the new MacBook Pro with the M3 chip and improved battery life.

2. Score: 0.1178
   Texte: Apple Inc. is a leading technology company known for the iPhone and MacBook.

3. Score: 0.0004
   Texte: Apple is a sweet and crisp fruit, often red or green, grown on trees.

💡 Analyse : Les documents sur Apple (entreprise) doivent avoir les scores les plus élevés.


Partie 2 : Configuration d’un index serverless pour des notes médicales

7 : Installation des bibliothèques pour les données et le ML

In [ ]:
# PARTIE 2 - ÉTAPE 1 : Installation de pandas, torch et transformers
# Ces librairies vont nous servir à charger les données,
# générer des embeddings avec un modèle SentenceTransformer.

!pip install -q pandas torch transformers

print("✅ Bibliothèques installées.")

✅ Bibliothèques installées.


8 : Imports et configuration de l'environnement

In [ ]:
# PARTIE 2 - ÉTAPE 2 : Imports et paramètres
# On définit le cloud (AWS), la région (us-east-1) et le nom de l'index.
# On utilise ServerlessSpec pour créer un index sans serveur.

import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Configuration du cloud et de la région (valeurs par défaut)
cloud = os.getenv('PINECONE_CLOUD', 'aws')         # Cloud provider
region = os.getenv('PINECONE_REGION', 'us-east-1') # Région AWS

# Spécifications serverless
spec = ServerlessSpec(cloud=cloud, region=region)

# Nom de l'index (choisis un nom explicite)
index_name = 'medical-notes-index'

print(f"☁️  Cloud : {cloud}")
print(f"🌍 Région : {region}")
print(f"📂 Nom de l'index : {index_name}")

☁️  Cloud : aws
🌍 Région : us-east-1
📂 Nom de l'index : medical-notes-index


9 : Création (ou recréation) de l'index

In [ ]:
# PARTIE 2 - ÉTAPE 3 : Création de l'index

# On supprime l'ancien index s'il existe pour éviter les conflits.
# Dimension = 384 (taille des vecteurs générés par all-MiniLM-L6-v2).
# Métrique = 'cosine' (similarité cosinus).

# Nettoyage de l'ancien index
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)
    print(f"🗑️  Ancien index '{index_name}' supprimé.")

# Création du nouvel index
pc.create_index(
    name=index_name,
    dimension=384,          # Taille des embeddings du modèle choisi
    metric='cosine',        # Métrique de similarité
    spec=spec
)

print(f"✅ Index '{index_name}' créé avec succès.")

✅ Index 'medical-notes-index' créé avec succès.


Partie 3 : Chargement des données d'exemple (notes médicales)

10 : Téléchargement du fichier JSONL

In [ ]:
# PARTIE 3 - ÉTAPE 1 : Téléchargement du dataset
# On utilise requests pour télécharger le fichier JSONL depuis GitHub.
# Ce fichier contient des notes médicales déjà pré-traitées.

import requests
import tempfile
import os # Import the os module here

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # URL brute du fichier sur GitHub (corrected path)
    url = "https://raw.githubusercontent.com/pinecone-io/examples/master/gen-ai/retrieval/medical-notes-reranking/data/sample_notes_data.jsonl"

    response = requests.get(url)
    response.raise_for_status()  # Lève une erreur si le téléchargement échoue

    with open(file_path, "wb") as f:
        f.write(response.content)

    # Chargement du fichier JSONL dans un DataFrame pandas
    df = pd.read_json(file_path, orient='records', lines=True)

print("✅ Données chargées avec succès.")

HTTPError: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/pinecone-io/examples/master/learn/generation/rag/rag-with-reranking/data/sample_notes_data.jsonl

11 : Aperçu du DataFrame

In [ ]:
# PARTIE 3 - ÉTAPE 2 : Inspection des données
# On vérifie la forme du DataFrame et on affiche les premières lignes.
# Les colonnes importantes sont : 'id', 'values' (les embeddings), et 'metadata'.

print("Data shape:", df.shape)  # Affiche (nombre de lignes, nombre de colonnes)
print("\n📊 Aperçu du DataFrame :")
df.head()

Partie 4 : Insertion (Upsert) des données dans l'index

12 : Upsert des vecteurs

In [ ]:
# PARTIE 4 - ÉTAPE 1 : Insertion des données
# On instancie le client pour l'index créé précédemment.
# La méthode upsert_from_dataframe() envoie les vecteurs et métadonnées à Pinecone.

index = pc.Index(name=index_name)

# Upsert : envoi du DataFrame dans l'index.
# Le DataFrame doit contenir 'id', 'values' (embeddings) et 'metadata'.
index.upsert_from_dataframe(df)

print("✅ Données insérées dans l'index.")

13 : Attente de la disponibilité de l'index

In [ ]:
# PARTIE 4 - ÉTAPE 2 : Vérification de la disponibilité
# On attend que les vecteurs soient indexés avant d'effectuer des requêtes.
# On vérifie que le nombre de vecteurs est > 0.

def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Nombre de vecteurs dans l'index : {vector_count}")
    return vector_count > 0  # On attend au moins 1 vecteur

print("⏳ Attente de l'indexation...")
while not is_fresh(index):
    time.sleep(5)  # On attend 5 secondes entre chaque vérification

print("✅ Index prêt !")
print("\n📊 Statistiques de l'index :")
index.describe_index_stats()

Partie 5 : Fonction d'embedding et requête sémantique

14 : Définition de la fonction d'embedding

In [ ]:
# PARTIE 5 - ÉTAPE 1 : Fonction get_embedding()
# On utilise le modèle 'sentence-transformers/all-MiniLM-L6-v2' pour encoder
# la requête dans le même espace vectoriel que nos notes médicales.

def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    # Tokenisation de la question
    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')

    with torch.no_grad():
        model_output = model(**encoded_input)

    # On moyenne les tokens sur la dimension de la séquence (dim=0)
    # pour obtenir un vecteur unique par phrase.
    embedding = model_output.last_hidden_state[0].mean(dim=0)

    return embedding

print("✅ Fonction d'embedding prête.")

15 : Exécution d'une recherche sémantique

In [ ]:
# PARTIE 5 - ÉTAPE 2 : Recherche sémantique
# On pose une question médicale, on génère son embedding,
# puis on interroge l'index Pinecone.

question = "Patient has severe chest pain radiating to the left arm"

# Génération de l'embedding de la question
query_vector = get_embedding(question).tolist()

# Recherche dans l'index (top_k = 5 résultats)
results = index.query(vector=[query_vector], top_k=5, include_metadata=True)

# Tri des résultats par score (ordre décroissant)
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

print(f"🔎 Requête : '{question}'")
print(f"✅ {len(sorted_matches)} résultats trouvés.")

Partie 6 : Affichage et reranking des notes cliniques

16 : Affichage des résultats initiaux

In [ ]:
# PARTIE 6 - ÉTAPE 1 : Affichage des résultats bruts
# On affiche les résultats de la recherche sémantique avant reranking.

def show_results(question, matches):
    print(f"Question: '{question}'")
    print("\nRésultats initiaux (recherche sémantique) :")
    print("-" * 60)
    for i, match in enumerate(matches):
        print(f"{i+1}. ID: {match['id']}")
        print(f"   Score: {match['score']:.4f}")          # Champ 'score'
        print(f"   Metadata: {match['metadata']}")        # Champ 'metadata'
        print('')

show_results(question, sorted_matches)

17 : Préparation des documents pour le reranking

In [ ]:
# PARTIE 6 - ÉTAPE 2 : Transformation des documents
# On crée un champ 'reranking_field' qui concatène toutes les métadonnées.
# Cela donne au reranker un contexte riche pour recalculer les scores.

transformed_documents = [
    {
        'id': match['id'],
        # On concatène toutes les paires clé:valeur des métadonnées
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]

print("✅ Documents transformés pour le reranking.")
print("📄 Exemple de 'reranking_field' :")
print(transformed_documents[0]['reranking_field'][:200] + "...")

18 : Exécution du reranking serverless

In [ ]:
# PARTIE 6 - ÉTAPE 3 : Reranking des résultats
# On utilise une requête plus spécifique pour affiner le reranking.
# On spécifie 'rank_fields' pour indiquer quel champ utiliser pour le score.

refined_query = "Chest pain radiating to arm, possible myocardial infarction"

reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],  # Champ utilisé pour le reranking
    top_n=3,                          # On ne garde que les 3 meilleurs
    return_documents=True,            # On inclut les documents dans la réponse
)

print("✅ Reranking effectué avec succès.")

19 : Affichage des résultats rerankés

In [ ]:
# PARTIE 6 - ÉTAPE 4 : Affichage des résultats rerankés
# On affiche les résultats après reranking pour comparer avec les résultats initiaux.

def show_reranked_results(question, matches):
    print(f"Question: '{question}'")
    print("\nRésultats après Reranking :")
    print("-" * 60)
    for i, match in enumerate(matches):
        # match.score : le score de similarité recalculé par le reranker
        # match.document.text : le contenu du champ 'reranking_field' (car return_documents=True)
        print(f"{i+1}. ID: {match.document.id}")
        print(f"   Score (reranké): {match.score:.4f}")
        print(f"   Contexte: {match.document.text[:150]}...")  # On tronque pour l'affichage
        print('')

# 'reranked_results.data' contient la liste des résultats rerankés
show_reranked_results(refined_query, reranked_results.data)

print("💡 Analyse : Les résultats rerankés sont mieux classés selon la pertinence clinique.")

20 : Nettoyage (optionnel)

In [ ]:
# PARTIE 6 - ÉTAPE 5 : Nettoyage
# Pour éviter des frais inutiles, on supprime l'index une fois l'exercice terminé.
# Décommente la ligne ci-dessous si tu veux supprimer l'index.

# pc.delete_index(name=index_name)
# print(f"🗑️  Index '{index_name}' supprimé.")

print("💡 N'oublie pas de supprimer ton index si tu ne l'utilises plus !")

Récapitulatif final et critères de réussite

In [ ]:
# RÉCAPITULATIF FINAL

print("=" * 60)
print(" CHALLENGE PINEAPPS : RERANKING SERVERLESS - TERMINÉ")
print("=" * 60)

print("""
 Critères de réussite validés :

1. Authentification Pinecone réussie
2. Reranking de documents de base exécuté et résultats affichés
3. Index serverless créé et peuplé avec des notes médicales
4. Recherche sémantique exécutée sur les données médicales
5. Résultats initiaux comparés aux résultats rerankés
6. Compréhension de l'amélioration de la pertinence par le reranking

 Ce que tu as appris :
- Configuration d'un index Pinecone serverless
- Génération d'embeddings avec Sentence Transformers
- Utilisation de l'API de reranking de Pinecone
- Construction d'un pipeline de recherche augmentée (RAG) avec reranking
- Différence entre recherche par similarité et reranking

 Prochaines étapes possibles :
- Tester avec différentes requêtes médicales
- Ajuster les paramètres top_n et rank_fields
- Intégrer ce pipeline dans une application de Q/A clinique
""")



 CHALLENGE PINEAPPS : RERANKING SERVERLESS - TERMINÉ

 Critères de réussite validés :

1. Authentification Pinecone réussie
2. Reranking de documents de base exécuté et résultats affichés
3. Index serverless créé et peuplé avec des notes médicales
4. Recherche sémantique exécutée sur les données médicales
5. Résultats initiaux comparés aux résultats rerankés
6. Compréhension de l'amélioration de la pertinence par le reranking

 Ce que tu as appris :
- Configuration d'un index Pinecone serverless
- Génération d'embeddings avec Sentence Transformers
- Utilisation de l'API de reranking de Pinecone
- Construction d'un pipeline de recherche augmentée (RAG) avec reranking
- Différence entre recherche par similarité et reranking

 Prochaines étapes possibles :
- Tester avec différentes requêtes médicales
- Ajuster les paramètres top_n et rank_fields
- Intégrer ce pipeline dans une application de Q/A clinique

